In [1]:
import pandas as pd
import seaborn 
import matplotlib.pyplot as plt 
import re
import glob
import os
import json

In [2]:
original_table_path = '/mnt/datalake/openmind/MedP-Midas/notebooks/eda/data/sessions.tsv'
# Base directory where all session folders reside
base_dir = '/mnt/datalake/openmind/MedP-Midas/dataset/MedP'
# Output path for the new consolidated table
output_path = '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/new_patient_table.csv'

# Load the original table into a DataFrame
# If your table is tab-separated, use sep='\t', otherwise adjust accordingly
df = pd.read_csv(original_table_path, sep='\t')

# Helper to find the .json file for a given filename prefix
def find_json_path(nii_path, base_dir):
    # 1) derive the prefix *without* .nii or .nii.gz
    if nii_path.endswith('.nii.gz'):
        prefix = os.path.basename(nii_path)[:-7]   # drop “.nii.gz”
    elif nii_path.endswith('.nii'):
        prefix = os.path.basename(nii_path)[:-4]   # drop “.nii”
    else:
        prefix = os.path.basename(nii_path)

    # 2) now look for prefix + ".json" anywhere under base_dir
    pattern = os.path.join(base_dir, '**', f'{prefix}.json')
    matches = glob.glob(pattern, recursive=True)
    return matches[0] if matches else None


# Prepare a list to collect new rows
rows = []

for idx, row in df.iterrows():
    prefix = row['filename']
    json_file = find_json_path(prefix, base_dir)
    dicom_date = None
    patient_id = None
    if json_file and os.path.exists(json_file):
        with open(json_file, 'r') as f:
            meta = json.load(f)
        # Extract patient ID (00100020) and acquisition date (00080020)
        patient_id = meta.get('00100020', {}).get('Value', [None])[0]
        dicom_date = meta.get('00080020', {}).get('Value', [None])[0]
    else:
        print(f"Warning: JSON file not found for prefix {prefix}")

    row_dict = {
        'filename': prefix,
        'patient_label': row['patient label'],
        'patient_id': patient_id,
        'image_date': dicom_date,
        'age': row["Patient's Age (00101010)"],
        'sex': row["Patient's Sex (00100040)"],
        'weight': row["Patient's Weight (00101030)"]
    }
    rows.append(row_dict)

    # Imprime cada 10 filas o la primera fila
    if idx < 3 or idx % 10 == 0:
        print(f"Fila {idx}: {row_dict}")

# ...existing code...

Fila 0: {'filename': 'sub-S0018518_ses-E0019865_run-1_bp-lsspine_vp-sag_T2w.nii.gz', 'patient_label': '260824140281153420865588499607617203044', 'patient_id': '260824140281153420865588499607617203044', 'image_date': '20160515', 'age': 56.0, 'sex': 'M', 'weight': 100.0}
Fila 1: {'filename': 'sub-S0018518_ses-E0019865_run-1_bp-lsspine_vp-sag_STIR.nii.gz', 'patient_label': '260824140281153420865588499607617203044', 'patient_id': '260824140281153420865588499607617203044', 'image_date': '20160515', 'age': 56.0, 'sex': 'M', 'weight': 100.0}
Fila 2: {'filename': 'sub-S0018518_ses-E0019865_acq5_run-1_bp-spine_vp-ax_T2w.nii.gz', 'patient_label': '260824140281153420865588499607617203044', 'patient_id': '260824140281153420865588499607617203044', 'image_date': '20160515', 'age': 56.0, 'sex': 'M', 'weight': 100.0}
Fila 10: {'filename': 'sub-S0018237_ses-E0019584_acq5_run-1_bp-spine_vp-ax_T2w.nii.gz', 'patient_label': '2622748596516663767697416823502566573', 'patient_id': '26227485965166637676974168

KeyboardInterrupt: 

In [ ]:
main_csv = '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/new_patient_table.csv'
diag_csv = '/mnt/datalake/openmind/MedP-Midas/variables/sd2022/SD2022_CONSULTAS_con_SIP_codif.txt'

df_main = pd.read_csv(main_csv)
df_diag = pd.read_csv(diag_csv, sep='|')

# Convierte las fechas a datetime
df_main['image_date'] = pd.to_datetime(df_main['image_date'], format='%Y%m%d', errors='coerce')
df_diag['FECHA'] = pd.to_datetime(df_diag['FECHA'], format='%Y%m%d', errors='coerce')

# Función para encontrar el diagnóstico más cercano en fecha
def get_closest_diag(row):
    patient = row['patient_label']
    img_date = row['image_date']
    df_patient = df_diag[df_diag['SIP_codif'] == str(patient)]
    if df_patient.empty or pd.isnull(img_date):
        return pd.Series([None, None, None])
    df_patient = df_patient.copy()
    df_patient['date_diff'] = (df_patient['FECHA'] - img_date).abs()
    closest = df_patient.loc[df_patient['date_diff'].idxmin()]
    return pd.Series([img_date, closest['FECHA'], closest['COD-DIAG'], closest['DESC-DIAG']])

# Aplica la función y añade las columnas
df_main[['image_date', 'closest_diag_date', 'diag_code', 'diag_desc']] = df_main.apply(get_closest_diag, axis=1)

# Guarda el resultado
df_main.to_csv('/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/new_patient_table_with_diag.csv', index=False)

# Muestra las primeras filas para comprobar
print(df_main[['filename', 'patient_label', 'image_date', 'closest_diag_date', 'closest_diag_code', 'closest_diag_desc']].head())